# Import Libraries

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import r2_score, accuracy_score, f1_score
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout

# Load Dataset

In [2]:
df = pd.read_csv('/content/water_quality.csv')

# Define features

In [3]:
feature_cols = ['pH', 'EC', 'CO3', 'HCO3', 'Cl', 'SO4', 'NO3', 'TH', 'Ca', 'Mg', 'Na', 'K', 'F', 'TDS']
X = df[feature_cols]

# Impute missing values and scale data

In [4]:
X = X.fillna(X.median())
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# REGRESSION MODEL (Predicting WQI)

In [6]:
y_wqi = df['WQI']
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(
    X_scaled, y_wqi, test_size=0.2, random_state=42
)

# Build Model

In [7]:
wqi_model = Sequential([
    Dense(64, activation='relu', input_shape=(X_scaled.shape[1],)),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(1, activation='linear')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


# Compile & Train

In [14]:
wqi_model.compile(optimizer='adam', loss='mse', metrics=['mae'])
wqi_model.fit(X_train_r, y_train_r, epochs=100, batch_size=32, validation_split=0.2, verbose=1)

Epoch 1/100
381/381 ━━━━━━━━━━━━━━━━━━━━ 3s 3ms/step - loss: 457.0883 - mae: 10.3643 - val_loss: 19.6253 - val_mae: 2.7702
Epoch 2/100
381/381 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 473.9006 - mae: 10.2834 - val_loss: 120.7617 - val_mae: 5.4875
Epoch 3/100
381/381 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 627.8815 - mae: 10.7028 - val_loss: 31.5704 - val_mae: 3.2179
Epoch 4/100
381/381 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 495.8916 - mae: 10.2512 - val_loss: 19.1239 - val_mae: 2.6058
Epoch 5/100
381/381 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 543.8726 - mae: 10.5817 - val_loss: 27.6590 - val_mae: 3.9023
Epoch 6/100
381/381 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 535.1999 - mae: 10.3570 - val_loss: 63.0363 - val_mae: 4.9663
Epoch 7/100
381/381 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - loss: 556.3064 - mae: 10.6293 - val_loss: 15.4461 - val_mae: 2.3053
Epoch 8/100
381/381 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - loss: 463.5478 - mae: 10.1837 - val_loss: 22.0529 - val_mae: 3.1444
Epoch 9/100
381

# Evaluate

In [10]:
wqi_preds = wqi_model.predict(X_test_r)
print(f"Regression Model R-squared Score: {r2_score(y_test_r, wqi_preds):.4f}")

119/119 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
Regression Model R-squared Score: 0.9996


# CLASSIFICATION MODEL (Predicting Category)

In [11]:
y_class = df['Water Quality Classification']
y_class_encoded = LabelEncoder().fit_transform(y_class)
X_train_c, X_test_c, y_train_c, y_test_c = train_test_split(X_scaled, y_class_encoded, test_size=0.2, random_state=42)

# Build Model

In [12]:
class_model = Sequential([
    Dense(64, activation='relu', input_shape=(X_scaled.shape[1],)),
    Dropout(0.2),
    Dense(32, activation='relu'),
    Dense(len(np.unique(y_class_encoded)), activation='softmax')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


# Compile & Train

In [13]:
class_model.compile(optimizer='adam', loss='sparse_categorical_crossentropy', metrics=['accuracy'])
class_model.fit(X_train_c, y_train_c, epochs=50, batch_size=32, validation_split=0.2, verbose=1)

Epoch 1/50
381/381 ━━━━━━━━━━━━━━━━━━━━ 6s 6ms/step - accuracy: 0.7431 - loss: 0.7125 - val_accuracy: 0.8713 - val_loss: 0.3522
Epoch 2/50
381/381 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.8721 - loss: 0.3254 - val_accuracy: 0.9117 - val_loss: 0.2302
Epoch 3/50
381/381 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9042 - loss: 0.2417 - val_accuracy: 0.9386 - val_loss: 0.1745
Epoch 4/50
381/381 ━━━━━━━━━━━━━━━━━━━━ 1s 3ms/step - accuracy: 0.9171 - loss: 0.2036 - val_accuracy: 0.9507 - val_loss: 0.1487
Epoch 5/50
381/381 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9259 - loss: 0.1787 - val_accuracy: 0.9550 - val_loss: 0.1316
Epoch 6/50
381/381 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9344 - loss: 0.1602 - val_accuracy: 0.9547 - val_loss: 0.1182
Epoch 7/50
381/381 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9385 - loss: 0.1462 - val_accuracy: 0.9626 - val_loss: 0.1102
Epoch 8/50
381/381 ━━━━━━━━━━━━━━━━━━━━ 1s 2ms/step - accuracy: 0.9452 - loss: 0.1329 - val_accuracy: 0.

# Evaluate

In [15]:
class_preds = np.argmax(class_model.predict(X_test_c), axis=1)
print(f"Classification Model Accuracy: {accuracy_score(y_test_c, class_preds):.4f}")
print(f"Classification Model F1-Score: {f1_score(y_test_c, class_preds, average='weighted'):.4f}")

119/119 ━━━━━━━━━━━━━━━━━━━━ 0s 1ms/step
Classification Model Accuracy: 0.9800
Classification Model F1-Score: 0.9800


# Testing Function

In [19]:
label_encoder = LabelEncoder()
label_encoder.fit(df['Water Quality Classification'])

def predict_water_quality(input_data):
    input_df = pd.DataFrame([input_data], columns=feature_cols)
    input_df_filled = input_df.fillna(X.median())
    input_scaled = scaler.transform(input_df_filled)

    predicted_wqi = wqi_model.predict(input_scaled)[0][0]
    class_probabilities = class_model.predict(input_scaled)
    predicted_class_idx = np.argmax(class_probabilities, axis=1)[0]
    predicted_class_label = label_encoder.inverse_transform([predicted_class_idx])[0]

    return predicted_wqi, predicted_class_label

### Prediction Example

In [20]:
new_location_data = {
    'pH': 7.5, 'EC': 300, 'CO3': 20, 'HCO3': 100, 'Cl': 30, 'SO4': 25,
    'NO3': 5, 'TH': 150, 'Ca': 40, 'Mg': 20, 'Na': 50, 'K': 3, 'F': 0.5, 'TDS': 200
}

predicted_wqi, predicted_classification = predict_water_quality(new_location_data)
print(f"Predicted WQI: {predicted_wqi:.2f}")
print(f"Predicted Classification: {predicted_classification}")

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 36ms/step
1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 37ms/step
Predicted WQI: 83.18
Predicted Classification: Good
